In [1]:
import os, time, subprocess, shutil, glob, json, threading

OUT = "/kaggle/working"
os.makedirs(OUT + "/repo", exist_ok=True)
TEMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
T_START = time.time()

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout

report = "GPU:\n" + sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
report += "\nCPU:\n" + sh("lscpu | grep -E 'Model name|^CPU\\(s\\)'")
report += "\nRAM:\n" + sh("free -h | head -2")
report += "\nDisk (ROTA 0 means SSD):\n" + sh("lsblk -d -o NAME,ROTA")
report += "\nOS:\n" + sh("grep PRETTY_NAME /etc/os-release")
report += "\nPython:\n" + sh("python --version")
report += "\nFree space:\n" + sh("df -h /kaggle/working " + TEMP + " /dev/shm")
with open(OUT + "/hardware_raw.txt", "w") as f:
    f.write(report)
print(report)

GPU:
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

CPU:
CPU(s):                                  4
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz

RAM:
               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.4Gi        24Gi       1.7Mi       5.6Gi        29Gi

Disk (ROTA 0 means SSD):
NAME  ROTA
loop0    0
loop1    0
loop2    0
sda      0
sdb      0

OS:
PRETTY_NAME="Ubuntu 24.04.4 LTS"

Python:
Python 3.13.16

Free space:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   68K   20G   1% /kaggle/working
overlay         8.0T  5.6T  2.5T  70% /
shm              14G     0   14G   0% /dev/shm



In [2]:
!pip install -q ultralytics huggingface_hub pyarrow

import importlib.metadata as md
import torch
lines = []
for name in ["ultralytics", "torch", "opencv-python", "opencv-python-headless",
             "huggingface_hub", "pyarrow", "numpy"]:
    try:
        lines.append(name + "==" + md.version(name))
    except Exception:
        pass
lines.append("torch cuda " + str(torch.version.cuda))
lines.append("gpus " + str(torch.cuda.device_count()))
with open(OUT + "/versions.txt", "w") as f:
    f.write("\n".join(lines))
print("\n".join(lines))

ultralytics==8.4.216
torch==2.11.0+cu130
opencv-python==4.14.0.94
opencv-python-headless==4.14.0.94
huggingface_hub==1.33.0
pyarrow==23.0.1
numpy==2.1.3
torch cuda 13.0
gpus 2


In [3]:
%%writefile /kaggle/working/repo/run.py
import argparse
import glob
import json
import logging
import os
import queue
import random
import shutil
import tempfile
import threading
import time

import cv2
import pyarrow.parquet as pq
from ultralytics import YOLO

logging.getLogger("ultralytics").setLevel(logging.ERROR)

CLASS_IDS = [0, 2, 16, 63, 67]
CLASS_NAMES = ["person", "car", "dog", "laptop", "cell phone"]
# (frames looked at first, frames when they disagree; 0 means every frame)
LADDER = [(2, 12), (3, 12), (4, 16), (3, 0)]
LIMITS = [3.0, 5.0, 1.0]  # alarm limits in percent: object flips, people misses, people off by 2+
WINDOW = 300
FAST_RATE = 0.06
SLOW_RATE = 0.03
GROUP = 8


def summarize(counts):
    # Objects present in at least half the frames, and the median people count
    n = len(counts)
    present = [0, 0, 0, 0, 0]
    if n == 0:
        return present, 0
    for c in range(5):
        seen = 0
        for row in counts:
            if row[c] > 0:
                seen += 1
        if 2 * seen >= n:
            present[c] = 1
    people = sorted([row[0] for row in counts])
    if n % 2 == 1:
        return present, people[n // 2]
    return present, (people[n // 2 - 1] + people[n // 2]) // 2


def disagree(counts):
    # True if the sampled frames do not agree on an object or on the people count
    if len(counts) < 2:
        return False
    for c in range(5):
        seen = 0
        for row in counts:
            if row[c] > 0:
                seen += 1
        if seen > 0 and seen < len(counts):
            return True
    for row in counts:
        if row[0] != counts[0][0]:
            return True
    return False


def run_model(model, frames):
    counts = []
    if len(frames) == 0:
        return counts
    results = model.predict(frames, imgsz=640, conf=0.4, iou=0.7, classes=CLASS_IDS,
                            half=True, batch=32, verbose=False)
    for r in results:
        ids = r.boxes.cls.tolist()
        row = []
        for cid in CLASS_IDS:
            row.append(ids.count(float(cid)))
        counts.append(row)
    return counts


class Shared:
    # The rung the decoder should use when it starts a new second
    def __init__(self, rung):
        self.rung = rung


def decode_video(data, vid, q, shared, tmp_dir):
    # Puts one packet per second on the queue; returns (error, frames decoded, frames reported)
    path = os.path.join(tmp_dir, "video_%d.mp4" % os.getpid())
    with open(path, "wb") as f:
        f.write(data)
    cap = cv2.VideoCapture(path)
    decoded = 0
    reported = 0
    error = None
    try:
        if not cap.isOpened():
            return "could not decode video", 0, 0
        cap.set(cv2.CAP_PROP_ORIENTATION_AUTO, 1)
        count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        if count and count > 0:
            reported = int(count)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if not fps or fps < 1 or fps > 240:
            fps = 25.0

        first_t = None
        last_t = 0.0
        cur = 0
        rung = shared.rung
        n0, m = LADDER[rung]
        j0 = 0
        jm = 0
        first = []
        extra = []
        while cap.grab():
            t = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
            if first_t is None:
                first_t = t
            elif t <= last_t:
                # timestamps that stand still or go backwards are pushed forward
                t = last_t + 1.0 / fps
            last_t = t
            decoded += 1
            rel = t - first_t
            sec = int(rel)
            if sec - cur > 3600:
                raise ValueError("timestamp jump")

            while cur < sec:
                q.put(("sec", vid, cur, first, extra, rung))
                cur += 1
                rung = shared.rung
                n0, m = LADDER[rung]
                j0 = 0
                jm = 0
                first = []
                extra = []

            take_first = False
            while j0 < n0 and rel >= cur + j0 / n0:
                j0 += 1
                take_first = True
            take_extra = False
            if m == 0:
                take_extra = True
            else:
                while jm < m and rel >= cur + jm / m:
                    jm += 1
                    take_extra = True

            if take_first or take_extra:
                ok, frame = cap.retrieve()
                if ok:
                    if take_first:
                        first.append(frame)
                    else:
                        extra.append(frame)
        if decoded > 0:
            q.put(("sec", vid, cur, first, extra, rung))
    except Exception as e:
        error = "decode failed: " + str(e)[:60]
    finally:
        cap.release()
        os.remove(path)
    return error, decoded, reported


def producer(files, done, q, shared, tmp_dir, max_videos):
    handled = 0
    for path in files:
        name = os.path.basename(path)
        try:
            pf = pq.ParquetFile(path)
            row = 0
            for batch in pf.iter_batches(batch_size=1, columns=["mp4"]):
                vid = name + ":" + str(row)
                row += 1
                if vid in done:
                    continue
                if max_videos > 0 and handled >= max_videos:
                    q.put(("done",))
                    return
                handled += 1
                data = batch.column(0)[0].as_py()
                if not data:
                    q.put(("end", vid, "no video data", 0, 0))
                    continue
                try:
                    error, decoded, reported = decode_video(data, vid, q, shared, tmp_dir)
                except Exception as e:
                    error, decoded, reported = "decode failed: " + str(e)[:60], 0, 0
                q.put(("end", vid, error, decoded, reported))
        except Exception as e:
            q.put(("end", name, "could not read parquet file: " + str(e)[:60], 0, 0))
    q.put(("done",))


def audit_rate(ctl):
    if len(ctl["window"]) < WINDOW:
        return FAST_RATE
    return SLOW_RATE


def check_alarm(ctl, shared):
    w = ctl["window"]
    if len(w) < WINDOW or ctl["rung"] >= len(LADDER) - 1:
        return
    for j in range(3):
        n = 0
        for flags in w:
            if flags[j]:
                n += 1
        if 100.0 * n / len(w) > LIMITS[j]:
            ctl["rung"] += 1
            shared.rung = ctl["rung"]
            ctl["window"] = []
            ctl["switches"].append((ctl["seconds"], LADDER[ctl["rung"]]))
            print("alarm: moving to", LADDER[ctl["rung"]], "after", ctl["seconds"], "seconds", flush=True)
            return


def process_group(model, group, ctl, rng, out):
    firsts = []
    for p in group:
        firsts.extend(p["first"])
    counts = run_model(model, firsts)

    pos = 0
    info = []
    extras = []
    for p in group:
        n = len(p["first"])
        mine = counts[pos:pos + n]
        pos += n
        m = LADDER[p["rung"]][1]
        escalate = disagree(mine)
        audit = False
        if m != 0 and p["rung"] == ctl["rung"]:
            if rng.random() < audit_rate(ctl):
                audit = True
        want = (escalate or audit) and len(p["extra"]) > 0
        if want:
            extras.extend(p["extra"])
        info.append([mine, escalate, audit, want])
    extra_counts = run_model(model, extras)
    ctl["frames"] += len(firsts) + len(extras)

    pos = 0
    for k in range(len(group)):
        p = group[k]
        mine, escalate, audit, want = info[k]
        everything = mine
        if want:
            n = len(p["extra"])
            everything = mine + extra_counts[pos:pos + n]
            pos += n
        present, people = summarize(everything)

        if audit:
            if escalate:
                ctl["window"].append([False, False, False])
            else:
                present1, people1 = summarize(mine)
                ctl["window"].append([present1 != present, people1 != people,
                                      abs(people1 - people) > 1])
            if len(ctl["window"]) > WINDOW:
                ctl["window"].pop(0)

        names = []
        for c in range(5):
            if present[c] == 1:
                names.append(CLASS_NAMES[c])
        out.write(json.dumps({"video": p["vid"], "second": p["sec"],
                              "objects": names, "people": people}) + "\n")
        ctl["seconds"] += 1


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True)
    ap.add_argument("--output", required=True)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--start-rung", type=int, default=0)
    ap.add_argument("--max-videos", type=int, default=0)
    args = ap.parse_args()

    start = time.time()
    files = sorted(glob.glob(os.path.join(args.input, "*.parquet")))
    done_path = args.output + ".done"
    state_path = args.output + ".state"

    done = set()
    rung = args.start_rung
    if args.resume and os.path.exists(done_path) and os.path.exists(args.output):
        for line in open(done_path):
            done.add(line.strip())
        if os.path.exists(state_path):
            rung = int(open(state_path).read().strip())
        # keep only the lines of finished videos
        kept = []
        for line in open(args.output):
            try:
                if json.loads(line)["video"] in done:
                    kept.append(line)
            except Exception:
                pass
        with open(args.output, "w") as f:
            f.writelines(kept)
        out = open(args.output, "a")
    else:
        out = open(args.output, "w")

    here = os.path.dirname(os.path.abspath(__file__))
    weights = os.path.join(here, "yolo11l.pt")
    if not os.path.exists(weights):
        weights = "yolo11l.pt"
    model = YOLO(weights)

    # /dev/shm can be tiny on some machines, so only use it when it has room
    tmp_dir = tempfile.gettempdir()
    if os.path.isdir("/dev/shm") and shutil.disk_usage("/dev/shm").free > 1000000000:
        tmp_dir = "/dev/shm"
    shared = Shared(rung)
    ctl = {"rung": rung, "window": [], "switches": [], "seconds": 0, "frames": 0}
    q = queue.Queue(maxsize=8)
    t = threading.Thread(target=producer, args=(files, done, q, shared, tmp_dir, args.max_videos),
                         daemon=True)
    t.start()

    rng = random.Random(0)
    pending = []
    failed = {"vid": None, "msg": None}
    videos_done = 0

    def flush():
        if len(pending) == 0:
            return
        group = list(pending)
        del pending[:]
        try:
            process_group(model, group, ctl, rng, out)
            check_alarm(ctl, shared)
        except Exception as e:
            failed["vid"] = group[0]["vid"]
            failed["msg"] = "model failed: " + str(e)[:60]

    while True:
        try:
            item = q.get_nowait()
        except queue.Empty:
            flush()
            item = q.get()

        if item[0] == "sec":
            if failed["vid"] == item[1]:
                continue
            pending.append({"vid": item[1], "sec": item[2], "first": item[3],
                            "extra": item[4], "rung": item[5]})
            if len(pending) >= GROUP:
                flush()
        elif item[0] == "end":
            flush()
            vid, error, decoded, reported = item[1], item[2], item[3], item[4]
            message = error
            if failed["vid"] == vid:
                message = failed["msg"]
            elif message is None:
                if decoded == 0:
                    message = "could not decode video"
                elif reported > 0 and decoded < 0.5 * reported:
                    message = "video stopped early"
            failed["vid"] = None
            if message is not None:
                out.write(json.dumps({"video": vid, "error": message}) + "\n")
            out.flush()
            if args.resume:
                with open(done_path, "a") as f:
                    f.write(vid + "\n")
                with open(state_path, "w") as f:
                    f.write(str(ctl["rung"]))
            videos_done += 1
            if videos_done % 20 == 0:
                spent = time.time() - start
                print("videos", videos_done, "| seconds", ctl["seconds"], "| model frames/s",
                      round(ctl["frames"] / spent), "| step", LADDER[ctl["rung"]],
                      "| elapsed", round(spent), "s", flush=True)
        else:
            flush()
            break

    out.close()
    spent = time.time() - start
    print("finished:", videos_done, "videos,", ctl["seconds"], "seconds,", round(spent), "s,",
          "switches", ctl["switches"], flush=True)


if __name__ == "__main__":
    main()

Writing /kaggle/working/repo/run.py


In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
DATA = TEMP + "/finevideo"
os.makedirs(DATA, exist_ok=True)

def parquet_name(i):
    return "train-%05d-of-01357.parquet" % i

def fetch(i):
    # Download one parquet file, trying a few times if the network hiccups
    for attempt in range(4):
        try:
            return hf_hub_download(repo_id="HuggingFaceFV/finevideo", repo_type="dataset",
                                   filename="data/" + parquet_name(i), local_dir=DATA,
                                   token=HF_TOKEN)
        except Exception as e:
            print("download problem on file", i, str(e)[:80], flush=True)
            time.sleep(10 * (attempt + 1))
    raise RuntimeError("could not download file " + str(i))

smoke_dir = TEMP + "/smoke"
os.makedirs(smoke_dir, exist_ok=True)
link = smoke_dir + "/" + parquet_name(0)
if not os.path.exists(link):
    os.symlink(fetch(0), link)

t0 = time.time()
r = subprocess.run(["python", "run.py", "--input", smoke_dir, "--output", OUT + "/smoke.jsonl",
                    "--max-videos", "3"], cwd=OUT + "/repo", capture_output=True, text=True)
print(r.stdout[-1500:])
print(r.stderr[-1500:])
print("smoke test seconds:", round(time.time() - t0))

lines = 0
errors = 0
for line in open(OUT + "/smoke.jsonl"):
    lines += 1
    if "error" in json.loads(line):
        errors += 1
print("smoke lines:", lines, "| error lines:", errors)
if lines < 100 or errors > 0:
    raise RuntimeError("smoke test failed, do not start the full run")
print(open(OUT + "/smoke.jsonl").readline())

data/train-00000-of-01357.parquet: reconstructing file:   0%|          |  0.00B /  494MB            

data/train-00000-of-01357.parquet: downloading bytes:           |  0.00B            

ettings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings

finished: 3 videos, 1082 seconds, 67 s, switches []


smoke test seconds: 77
smoke lines: 1082 | error lines: 0
{"video": "train-00000-of-01357.parquet:0", "second": 0, "objects": [], "people": 0}



In [ ]:
import gzip

# Both copies of run.py run at the same time, one per GPU
status = {}
expected = {}

def fetch_many(numbers):
    for i in numbers:
        fetch(i)

def worker(gpu, numbers):
    out_path = OUT + "/results_gpu%d.jsonl" % gpu
    log_path = OUT + "/log_gpu%d.txt" % gpu
    chunk_dir = TEMP + "/chunk_gpu%d" % gpu
    chunks = []
    for k in range(0, len(numbers), 5):
        chunks.append(numbers[k:k + 5])
    env = dict(os.environ)
    env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    try:
        fetch_many(chunks[0])
        for k in range(len(chunks)):
            chunk = chunks[k]
            prefetch = None
            if k + 1 < len(chunks):
                prefetch = threading.Thread(target=fetch_many, args=(chunks[k + 1],))
                prefetch.start()

            shutil.rmtree(chunk_dir, ignore_errors=True)
            os.makedirs(chunk_dir)
            for i in chunk:
                path = DATA + "/data/" + parquet_name(i)
                expected[i] = pq.ParquetFile(path).metadata.num_rows
                os.symlink(path, chunk_dir + "/" + parquet_name(i))

            t1 = time.time()
            ok = False
            for attempt in range(3):
                with open(log_path, "a") as log:
                    log.write("chunk " + str(chunk) + " attempt " + str(attempt) + "\n")
                    log.flush()
                    rc = subprocess.call(["python", "run.py", "--input", chunk_dir, "--output",
                                          out_path, "--resume"], cwd=OUT + "/repo",
                                         stdout=log, stderr=subprocess.STDOUT, env=env)
                if rc == 0:
                    ok = True
                    break
            with open(log_path, "a") as log:
                log.write("CHUNK " + str(chunk) + " ok=" + str(ok) + " minutes " +
                          str(round((time.time() - t1) / 60, 1)) + "\n")

            if prefetch is not None:
                prefetch.join()
            for i in chunk:
                os.remove(DATA + "/data/" + parquet_name(i))
        status[gpu] = "finished"
    except Exception as e:
        status[gpu] = "crashed: " + str(e)[:200]

def last_line(path):
    if not os.path.exists(path):
        return "(no log yet)"
    lines = open(path).read().strip().split("\n")
    return lines[-1]

threads = []
threads.append(threading.Thread(target=worker, args=(0, list(range(0, 25)))))
threads.append(threading.Thread(target=worker, args=(1, list(range(25, 50)))))
for th in threads:
    th.start()
while threads[0].is_alive() or threads[1].is_alive():
    time.sleep(300)
    print(time.strftime("%H:%M"), "gpu0:", last_line(OUT + "/log_gpu0.txt"),
          "| gpu1:", last_line(OUT + "/log_gpu1.txt"), flush=True)
print("status:", status, flush=True)

# Merge the two result files and compress
total_lines = 0
error_lines = 0
seen_videos = set()
with open(OUT + "/results.jsonl", "w") as merged:
    for gpu in [0, 1]:
        path = OUT + "/results_gpu%d.jsonl" % gpu
        if not os.path.exists(path):
            continue
        for line in open(path):
            merged.write(line)
            total_lines += 1
            d = json.loads(line)
            seen_videos.add(d["video"])
            if "error" in d:
                error_lines += 1

with gzip.open(OUT + "/results.jsonl.gz", "wt") as gz:
    for line in open(OUT + "/results.jsonl"):
        gz.write(line)

minutes = round((time.time() - T_START) / 60, 1)
expected_total = sum(expected.values())
summary = ("total minutes (whole notebook): " + str(minutes) + "\n" +
           "lines: " + str(total_lines) + "\n" +
           "videos with a line: " + str(len(seen_videos)) + " of " + str(expected_total) + "\n" +
           "error lines: " + str(error_lines) + "\n" +
           "status: " + str(status) + "\n")
with open(OUT + "/run_summary.txt", "w") as f:
    f.write(summary)
print(summary)